# Trinity Finetuning — Unsloth (LoRA)

Finetunes `unsloth/Llama-3.2-3B-Instruct` with LoRA on a small cyber-defense
instruction dataset using [Unsloth](https://github.com/unslothai/unsloth) for
fast, memory-efficient training.

**Run this in Google Colab with a T4 GPU runtime** (Runtime → Change runtime
type → T4 GPU). This Mac has no CUDA GPU, so Unsloth cannot run locally.

## How to run this notebook in Colab

1. Go to [colab.research.google.com](https://colab.research.google.com/) →
   **File → Upload notebook** → select `unsloth_finetune.ipynb` from
   `notebooks/unsloth/` in this repo.
2. **Runtime → Change runtime type → T4 GPU** (or better), then Save.
3. In the Colab file browser (folder icon, left sidebar), click **Upload**
   and add `cyber_defense_qa.jsonl` from `notebooks/data/` in this repo.
4. **Runtime → Run all**. Training (60 steps) should take a few minutes on a
   T4.

Steps: install deps → load 4-bit model → attach LoRA adapters → format the
`cyber_defense_qa.jsonl` dataset → train with `trl`'s `SFTTrainer` → compare
base vs. finetuned output → save the adapter → notes on exporting to GGUF for
Ollama.


## 1. Install dependencies

Colab ships with an old `torch`; Unsloth's installer handles compatible
versions of `torch`, `xformers`, `trl`, `peft`, and `bitsandbytes`.


In [ ]:
%%capture
import importlib.util

if importlib.util.find_spec("google.colab") is not None:
    # Colab-specific install known to work with Unsloth's dependency pins.
    %pip install unsloth
else:
    print("Not running in Colab — skipping install. Run this notebook in Colab with a GPU runtime.")


## 2. Configuration


In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

DATASET_PATH = "cyber_defense_qa.jsonl"  # upload this file into the Colab session first (see instructions above)

LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

OUTPUT_DIR = "trinity-unsloth-lora"
MAX_STEPS = 60  # small smoke-test run; increase for a real training pass
LEARNING_RATE = 2e-4

SYSTEM_PROMPT = (
    "You are Trinity, a cybersecurity defense assistant. You help with threat "
    "detection, incident response, secure coding, vulnerability analysis, and "
    "hardening systems. Be concise, precise, and security-conscious."
)


## 3. Load the base model (4-bit) and attach LoRA adapters


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


## 4. Load and format the dataset

Formats each `{instruction, response}` pair into the model's chat template
with the Trinity system prompt.


In [ ]:
import os

from datasets import load_dataset

assert os.path.exists(DATASET_PATH), (
    f"{DATASET_PATH} not found — upload it into the Colab session first (see instructions above)."
)

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")


def format_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
print(dataset)
print(dataset[0]["text"])


## 5. Train with `trl`'s `SFTTrainer`


In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
    ),
)

trainer_stats = trainer.train()


## 6. Quick inference test (base vs. finetuned)


In [ ]:
FastLanguageModel.for_inference(model)

test_question = "What should I check first when a server shows unexpected outbound traffic?"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_question},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))


## 7. Save the LoRA adapter


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")


## 8. Export to GGUF for Ollama (optional)

To use the finetuned model with the existing `finetune-poc` Ollama setup,
merge the adapter and export to GGUF, then reference it from a new
`FROM ./trinity-unsloth.gguf` line in an updated Modelfile:

```python
model.save_pretrained_gguf(
    "trinity-unsloth-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
```

Download the resulting `.gguf` file from Colab and copy it into the
`ollama/` directory locally, then update `ollama/Modelfile` to build from it
instead of `FROM llama3.2`.
